# Session 1: a flexible curve that can also obey a law
**A. Real-world problem.** A cooling cup of tea is measured six times in its first minute.
The readings are a little noisy. You must predict the temperature excess for the whole next
minute too. A flexible curve can pass near all six readings and still swing wildly where you
have no readings. But you also know a law: the excess temperature falls at a rate proportional
to itself, `u' = -u` (we use scaled units). Can a curve be asked to obey the law as well as
the data?

**Plan.** (1) build a flexible curve from "tanh bumps", (2) fit it to the six readings, (3) add
the law as extra rows in the same fit, (4) measure the difference.

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

STEEP = 3.0                                    # how sharp each bump is
DATA_X = np.array([0.0, 0.2, 0.4, 0.6, 0.8, 1.0])           # times of the six readings
NOISE = np.array([0.02, -0.03, 0.025, -0.02, 0.03, -0.025])   # fixed "measurement errors"
DATA_Y = np.exp(-DATA_X) + NOISE                # the noisy readings

## Step 1: one bump, then many bumps
`tanh(s)` is a smooth step: it is -1 far to the left, +1 far to the right and passes through
0 at s = 0. Shift it with `x - centre` and sharpen it with STEEP. A "unit" of a network is one
such bump; the curve is a weighted sum of units plus a constant:

    u(x) = c_1 * tanh(STEEP (x - m_1)) + ... + c_H * tanh(STEEP (x - m_H)) + c_0

Here the centres m_j are spread evenly over 0 to 2 and we only choose the weights c.
The slope of the curve is also a sum: d tanh(s)/ds = 1 - tanh(s)^2.

In [ ]:
def features(x, n_units):
    """Return (F, D): the bump values and their slopes at every x. The last column is the constant."""
    x = np.asarray(x, dtype=float)
    centres = np.linspace(0.0, 2.0, n_units)
    th = np.tanh(STEEP * (x[:, None] - centres))
    ones = np.ones((len(x), 1))
    F = np.hstack([th, ones])
    D = np.hstack([STEEP * (1.0 - th ** 2), np.zeros((len(x), 1))])
    return F, D

## Step 2: fit the six readings (data only)
We choose weights c that make the curve close to the readings, by least squares: minimise
the sum of (curve - reading)^2. A tiny penalty on the size of c (the "ridge", 0.001) keeps
the sums sensible when there are more weights than readings.

In [ ]:
RIDGE = 1e-3


def fit(n_units, lam, n_law=21):
    """Weights c. lam = 0: data only. lam > 0: also ask u' + u = 0 at n_law points on [0, 2]."""
    F, _ = features(DATA_X, n_units)
    A = F.T @ F
    b = F.T @ DATA_Y
    if lam > 0:
        xs = np.linspace(0.0, 2.0, n_law)
        Fl, Dl = features(xs, n_units)
        R = Dl + Fl                    # row k gives u'(x_k) + u(x_k), which the law says is 0
        A = A + lam * (R.T @ R)
    A = A + RIDGE * np.eye(n_units + 1)
    return np.linalg.solve(A, b)

## Step 3: add the law
The law `u' = -u` says that at every time, slope + value = 0. We add "law rows" at 21 points
between 0 and 2 (the second minute included, where we have no readings!). `lam` says how
strongly we insist on the law compared with the data. Because the curve is a sum of bumps,
slope + value is again a weighted sum, so the whole problem is still one linear fit.

In [ ]:
def curve(c, x):
    F, _ = features(x, len(c) - 1)
    return F @ c


def law_residual(c, x):
    """u'(x) + u(x): zero everywhere if the curve obeys the law."""
    F, D = features(x, len(c) - 1)
    return (D + F) @ c


def rms_error(c, n_points=201):
    """Root-mean-square distance from the true curve exp(-x) over the whole range 0 to 2."""
    x = np.linspace(0.0, 2.0, n_points)
    return float(np.sqrt(np.mean((curve(c, x) - np.exp(-x)) ** 2)))

## Step 4: measure
The "true" answer exp(-x) is known here, so we can score both fits over the whole range
0 to 2, not only where the readings are.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    H = 12
    c_plain, c_law = fit(H, 0.0), fit(H, 1.0)
    print(f"{H} bumps, data only:    rms error over 0..2 = {rms_error(c_plain):.4f}")
    print(f"{H} bumps, data + law:   rms error over 0..2 = {rms_error(c_law):.4f}")
    xs = np.linspace(0, 2, 201)
    print(f"largest |u' + u| over 0..2: data only {np.abs(law_residual(c_plain, xs)).max():.3f}, "
          f"with law {np.abs(law_residual(c_law, xs)).max():.3f}")

    fig, ax = plt.subplots(figsize=(7, 4.2))
    ax.plot(xs, np.exp(-xs), "k--", label="true curve exp(-x)")
    ax.plot(xs, curve(c_plain, xs), color="tab:red", label="fit to readings only")
    ax.plot(xs, curve(c_law, xs), color="tab:green", label="fit to readings + law u' = -u")
    ax.plot(DATA_X, DATA_Y, "ko", label="six noisy readings")
    ax.set_ylim(-1.5, 2.0)
    ax.set_xlabel("time x (scaled units)")
    ax.set_ylabel("temperature excess u (scaled units)")
    ax.set_title("A flexible curve with and without the law")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "fit_with_without_law.png", dpi=120)

    # the bumps behind one fit
    F, _ = features(xs, 6)
    c6 = fit(6, 1.0)
    fig2, ax2 = plt.subplots(figsize=(7, 4.2))
    for j in range(6):
        ax2.plot(xs, c6[j] * F[:, j], lw=1, alpha=0.7, label=f"unit {j + 1}" if j < 3 else None)
    ax2.plot(xs, curve(c6, xs), "k", lw=2, label="sum of units + constant")
    ax2.set_xlabel("time x (scaled units)")
    ax2.set_ylabel("weighted bump value (scaled units)")
    ax2.set_title("A curve built from 6 weighted tanh bumps (law fit)")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "bumps_sum.png", dpi=120)

    print("\nrms error over 0..2 for different numbers of bumps and law strengths:")
    for n in (3, 6, 12):
        print(n, [round(rms_error(fit(n, lam)), 4) for lam in (0.0, 1.0, 10.0)])

## Where this stops
Here the bumps were fixed in place and only the weights were chosen. A real neural network also
moves the bumps. That needs gradients and backpropagation: Session 2.

## Coding tasks
1. **Easy.** Change the number of bumps in `lesson.py` from 12 to 5 and print the data-only and the law error; predict first whether the data-only error rises or falls.
2. **Stretch.** Add a seventh reading at x = 1.8 (use exp(-1.8) plus noise 0.02) to the readings and see how much the data-only fit improves; compare with adding the law.
3. **Explain.** Explain in a few sentences why the law rows are placed on the range 0 to 2, although the readings cover only 0 to 1.

## Assessment
- **A. Conceptual.** Why can a curve that passes within 0.05 of all six readings still be a poor prediction for the second minute?
- **B. Numerical.** A candidate curve is u = exp(-0.9 x). Compute the law residual u' + u at x = 1. Is the candidate a solution of u' = -u?
- **C. Graph and simulation interpretation.** In the app, with 6 units, compare the red and green curves at x = 2. What does the difference tell you, and what is the error with the law on?
- **D. Design and optimisation challenge.** Design: you can afford 6 units at most and want the rms error below 0.01. Choose the law weight, and say what happens if it is too large or if you use only 3 units.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    F, D = L.features(np.array([0.0, 1.0]), 4)
    checks.append(("features: shape is (points, units + 1) and the last column is 1", F.shape == (2, 5) and bool(np.all(F[:, -1] == 1.0))))
    # tanh passes through 0 at its centre (unit 1 is centred at x = 0)
    checks.append(("a tanh bump is 0 at its own centre", abs(F[0, 0]) < 1e-12))
    # slope of each bump against a finite difference
    x = np.array([0.3, 0.9, 1.4])
    h = 1e-6
    Fp, _ = L.features(x + h, 5)
    Fm, _ = L.features(x - h, 5)
    _, D = L.features(x, 5)
    checks.append(("hand-written bump slope matches a finite difference", np.allclose((Fp - Fm) / (2 * h), D, atol=1e-6)))

    c_plain, c_law = L.fit(12, 0.0), L.fit(12, 1.0)
    # worked example: the numbers quoted in Section E
    e_plain, e_law = L.rms_error(c_plain), L.rms_error(c_law)
    checks.append((f"worked example: data-only fit has rms error above 0.5 over 0..2 (got {e_plain:.3f})", e_plain > 0.5))
    checks.append((f"worked example: fit with the law has rms error below 0.001 (got {e_law:.4f})", e_law < 1e-3))
    xs = np.linspace(0, 2, 201)
    rp, rl = np.abs(L.law_residual(c_plain, xs)).max(), np.abs(L.law_residual(c_law, xs)).max()
    checks.append((f"largest |u' + u| is above 2 without the law and below 0.03 with it (got {rp:.2f}, {rl:.3f})", rp > 2.0 and rl < 0.03))

    # the data-only fit still passes close to the six readings
    F6, _ = L.features(L.DATA_X, 12)
    checks.append(("data-only fit passes within 0.05 of every reading", bool(np.abs(F6 @ c_plain - L.DATA_Y).max() < 0.05)))
    # hand calculation of the readings: exp(-0.4) + 0.025
    checks.append(("hand calculation: reading at x = 0.4 is exp(-0.4) + 0.025 = 0.695", abs(L.DATA_Y[2] - 0.695) < 1e-3))
    # edge: lam = 0 is exactly the data-only fit; one unit still works
    checks.append(("edge: lam = 0 gives the same weights as the data-only fit", np.allclose(L.fit(6, 0.0), L.fit(6, 0.0)) and L.fit(1, 1.0).shape == (2,)))
    # the law fit with 6 bumps beats the data-only fit, and too strong a law with only 3 bumps hurts
    checks.append(("6 bumps with law (lam 1) beat 6 bumps without", L.rms_error(L.fit(6, 1.0)) < 0.1 * L.rms_error(L.fit(6, 0.0))))
    checks.append(("trade-off: 3 bumps cannot obey the law well (error stays above 0.05 for lam 1)", L.rms_error(L.fit(3, 1.0)) > 0.05))
    checks.append(("too strong a law (lam 50) with 6 bumps is worse than lam 1", L.rms_error(L.fit(6, 50.0)) > L.rms_error(L.fit(6, 1.0))))
    # app challenge region
    checks.append(("app challenge: 6 to 16 bumps with lam 0.5 to 5 give rms below 0.01",
                   all(L.rms_error(L.fit(n, lam)) < 0.01 for n in (6, 8, 12, 16) for lam in (0.5, 1.0, 2.0, 5.0))))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
